# Subscription Migration to OneBill — Simplified Pipeline

Imports every subscription from `bi_curated_views.reporting_subscription` into
OneBill as a new order, one `POST /rest/OrderService/v1/order` per
subscription.

## Sources

| Data | Source | Purpose in payload |
|---|---|---|
| **Subscriptions** | MySQL `bi_curated_views.reporting_subscription` | `accountNumber`, `fulfilledDate`, `subscriptionIdentifier` |
| **Default shipping address** | OneBill `GET /rest/SubscriberService/v1/subscribers/{accountNumber}` | `shipAddId` |

## Key decisions (per your confirmation)

1. **`productName` / `priceplanName`** are hardcoded to `"SMS"` / `"SMS 50"` —
   the only plan currently set up in OneBill. Update
   `DEFAULT_PRODUCT_NAME` / `DEFAULT_PRICEPLAN_NAME` below once more plans exist.
2. **`recurringStartDate`** is always the 1st of the *current* month, not
   derived from the subscription's own start date.
3. **`shipAddId`** is looked up per-account from OneBill itself: the
   subscriber detail endpoint returns an `address` array, and we take the
   entry where `defaultShipping` is `true`. This is looked up once per
   account and cached, since several subscriptions can share an account.
4. **`actionType`** is always `"New"`, **`quantity`** is always `1`.

There are no other per-row transforms — every subscription in the view is
migrated as-is.

## Execution order

| Section | What happens |
|---|---|
| 1 | Imports and config |
| 2 | MySQL subscription query |
| 3 | OneBill token manager |
| 4 | Default shipping address lookup (cached per account) |
| 5 | Payload builder |
| 6 | POST worker |
| 7 | Parallel migration |
| 8 | Results, failures, error summary |

## 1. Imports and Configuration

Same `.env` keys as the account migration notebook, minus the Dataverse/CRM
block (subscriptions don't need contacts):

| Variable | Purpose |
|---|---|
| `DB_USERNAME` / `DB_PASSWORD` / `DB_HOST` | MySQL access |
| `CLIENT_ID` / `CLIENT_SECRET` / `API_USERNAME` / `API_PASSWORD` | OneBill OAuth |
| `CREATION_PROXY_ACCOUNT_NUMBER` | OneBill proxy account header value |

`load_dotenv(override=True)` makes the `.env` authoritative over any
pre-existing shell env — important when re-running after rotating a secret.

In [ ]:
# %pip install mysql-connector-python sqlalchemy python-dotenv requests pandas

import os
import time
import logging
import threading
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv(override=True)

# --- MySQL ---
BI_DATASTORE_URL = (
    f"mysql+mysqlconnector://{os.environ['DB_USERNAME']}:{os.environ['DB_PASSWORD']}"
    f"@{os.environ['DB_HOST']}/bi_datastore"
)

# --- OneBill ---
ONEBILL_BASE_URL      = "https://sandbox-sg.onebillsoftware.com"
ONEBILL_TOKEN_URL     = f"{ONEBILL_BASE_URL}/oauth/token"
ONEBILL_ORDER_URL     = f"{ONEBILL_BASE_URL}/rest/OrderService/v1/order"
ONEBILL_SUBSCRIBER_URL = f"{ONEBILL_BASE_URL}/rest/SubscriberService/v1/subscribers"  # + /{accountNumber}
ONEBILL_PROXY_ACCT    = os.environ["CREATION_PROXY_ACCOUNT_NUMBER"]

# --- Migration tunables ---
MAX_WORKERS        = 20
TOKEN_TTL_FALLBACK = 3500   # seconds; used only if OAuth response omits expires_in

# --- Fixed order fields (per current OneBill catalog — only one plan exists today) ---
DEFAULT_ORDER_STATE      = "1005"
DEFAULT_PRODUCT_NAME     = "SMS"
DEFAULT_PRICEPLAN_NAME   = "SMS 50"
DEFAULT_ACTION_TYPE      = "New"
DEFAULT_QUANTITY         = 1

# --- Logging ---
log_filename = f'subscription_migration_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log'
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.FileHandler(log_filename), logging.StreamHandler()],
)
logger = logging.getLogger(__name__)

## 2. MySQL Subscription Query

Pulls every row from `bi_curated_views.reporting_subscription` as-is — no
transforms, no filtering. `df_subscriptions.head(N)` below is only there so
you can test against a handful of rows before running the full set; remove
it (or raise `N`) once you're ready for the real migration.

In [ ]:
SUBSCRIPTION_QUERY = """
SELECT * FROM bi_curated_views.reporting_subscription;
""".strip()

engine = create_engine(BI_DATASTORE_URL)
df_subscriptions = pd.read_sql(SUBSCRIPTION_QUERY, con=engine)

logger.info(f"Loaded {len(df_subscriptions):,} subscriptions from MySQL")

df_subscriptions = df_subscriptions.head(5)  # Use this to limit rows while testing. Remove or increase when ready for a full run.

df_subscriptions.head()

## 3. OneBill Token Manager (Thread-Safe)

Identical to the account migration notebook — one shared bearer token across
all worker threads, refreshed proactively ~100 s before expiry.

In [ ]:
class TokenManager:
    """Thread-safe bearer token cache with proactive refresh."""

    def __init__(self):
        self._lock = threading.Lock()
        self._token: str | None = None
        self._expires_at: datetime = datetime.min

    def get_token(self) -> str:
        with self._lock:
            if datetime.now() >= self._expires_at:
                self._refresh()
            return self._token

    def _refresh(self) -> None:
        logger.info("Refreshing OneBill OAuth token...")
        token_data = {
            "grant_type":    "password",
            "client_id":     os.environ["CLIENT_ID"],
            "client_secret": os.environ["CLIENT_SECRET"],
            "username":      os.environ["API_USERNAME"],
            "password":      os.environ["API_PASSWORD"],
        }
        response = requests.post(
            ONEBILL_TOKEN_URL,
            data=token_data,
            headers={"Content-Type": "application/x-www-form-urlencoded"},
            timeout=30,
        )
        response.raise_for_status()
        payload = response.json()
        self._token = payload["access_token"]
        ttl = payload.get("expires_in", TOKEN_TTL_FALLBACK)
        # Refresh 100 s early to absorb clock skew + in-flight requests
        self._expires_at = datetime.now() + timedelta(seconds=ttl - 100)
        logger.info("Token valid until %s", self._expires_at.strftime("%H:%M:%S"))


token_manager = TokenManager()

## 4. Default Shipping Address Lookup

`shipAddId` isn't in the MySQL subscription data — it has to come from
OneBill itself. For a given `accountNumber`, we call:

```
GET /rest/SubscriberService/v1/subscribers/{accountNumber}
```

and take the `id` of the entry in the returned `address` array where
`"defaultShipping": true`.

Since multiple subscriptions in the migration can belong to the same
account, results are cached in `_ship_address_cache` so each account is only
looked up once, no matter how many of its subscriptions we're migrating. The
cache is a plain dict guarded by a lock; if two threads race to look up the
same brand-new account simultaneously, both may fetch once, but they'll
agree on the same address id, so it's a harmless duplicate GET rather than a
correctness issue.

In [ ]:
_ship_address_cache: dict[str, str] = {}
_ship_address_cache_lock = threading.Lock()


def get_default_ship_address_id(session: requests.Session, account_number: str) -> str:
    """Return the account's defaultShipping address id, fetching + caching on first use."""
    with _ship_address_cache_lock:
        cached = _ship_address_cache.get(account_number)
    if cached is not None:
        return cached

    headers = {"Authorization": f"Bearer {token_manager.get_token()}"}
    url = f"{ONEBILL_SUBSCRIBER_URL}/{account_number}"
    response = session.get(url, headers=headers, timeout=30)
    response.raise_for_status()
    detail = response.json()

    addresses = detail.get("address", [])
    default_address = next((a for a in addresses if a.get("defaultShipping")), None)
    if default_address is None:
        raise ValueError(f"No defaultShipping address found for account {account_number}")

    ship_add_id = str(default_address["id"])
    with _ship_address_cache_lock:
        _ship_address_cache[account_number] = ship_add_id
    return ship_add_id

## 5. Payload Builder

Builds the JSON body matching the OneBill order-create example exactly:

```
{
  "accountNumber":        <AccountCode>,
  "orderState":           "1005",
  "billThissOrder":       false,
  "isSkipProvisioning":   true,
  "orderElement": [
    {
      "quantity":               1,
      "actionType":             "New",
      "fulfilledDate":          <SubscriptionStartDate>,
      "recurringStartDate":     <1st of current month>,
      "subscriptionIdentifier": <SubscriptionUSN>,
      "productName":            "SMS",
      "priceplanName":          "SMS 50",
      "shipAddId":              <account's defaultShipping address id>
    }
  ]
}
```

`ship_add_id` is passed in (already resolved via the lookup above) rather
than hardcoded — everything else is fixed per your confirmation, with no
other per-row transforms.

In [ ]:
def _to_iso_midnight(value) -> str | None:
    """Format a date/datetime/str value as OneBill's 'YYYY-MM-DDT00:00:00'. None-safe."""
    if value is None or pd.isna(value):
        return None
    ts = pd.Timestamp(value)
    return ts.strftime("%Y-%m-%dT00:00:00")


def _beginning_of_this_month() -> str:
    """Return today's month, day 1, at midnight, in OneBill's date format."""
    today = datetime.now()
    return today.replace(day=1).strftime("%Y-%m-%dT00:00:00")


def build_subscription_payload(row: pd.Series, ship_add_id: str) -> dict:
    """Build the OneBill order-create payload for a single subscription row. No transforms."""
    return {
        "accountNumber":      str(row["AccountCode"]),
        "orderState":         DEFAULT_ORDER_STATE,
        "billThissOrder":     False,
        "isSkipProvisioning": True,
        "orderElement": [
            {
                "quantity":               DEFAULT_QUANTITY,
                "actionType":             DEFAULT_ACTION_TYPE,
                "fulfilledDate":          _to_iso_midnight(row["SubscriptionStartDate"]),
                "recurringStartDate":     _beginning_of_this_month(),
                "subscriptionIdentifier": str(row["SubscriptionUSN"]),
                "productName":            DEFAULT_PRODUCT_NAME,
                "priceplanName":          DEFAULT_PRICEPLAN_NAME,
                "shipAddId":              ship_add_id,
            }
        ],
    }

## 6. POST to OneBill + Per-Row Worker

`create_onebill_order` performs the actual POST and treats
`validationResponse.successful = False` as a failure even when the HTTP
status is 200 — OneBill returns business-logic errors that way (same
convention as the account migration notebook).

`migrate_row` is the per-subscription worker. It now does three timed steps:
resolve `shipAddId` (cached after the first subscription per account) →
build the payload → POST it. If the address lookup fails (e.g. no
`defaultShipping` address on the account), the row is marked failed and no
order is submitted for it.

In [ ]:
def create_onebill_order(session: requests.Session, order_url: str, payload: dict) -> dict:
    """POST one subscription/order to OneBill. Raises ValueError on validation failure."""
    headers = {"Authorization": f"Bearer {token_manager.get_token()}"}

    response = session.post(order_url, headers=headers, json=payload, timeout=30)
    response.raise_for_status()
    data = response.json()

    validation = data.get("validationResponse", {})
    if not validation.get("successful", True):
        errors   = validation.get("validationErrorInfo", [])
        messages = "; ".join(e.get("message", "") for e in errors)
        raise ValueError(messages or "validationResponse.successful = false")

    return data


def migrate_row(row: pd.Series, session: requests.Session) -> dict:
    """Resolve shipAddId, build, and POST a single subscription. Returns a result dict."""
    account_code     = row["AccountCode"]
    subscription_id  = row["SubscriptionUSN"]

    t_lookup, t_build, t_net = 0.0, 0.0, 0.0
    status, error, order_id, ship_add_id = "failed", None, None, None

    try:
        t0 = time.perf_counter()
        ship_add_id = get_default_ship_address_id(session, account_code)
        t_lookup = time.perf_counter() - t0

        t1 = time.perf_counter()
        payload = build_subscription_payload(row, ship_add_id)
        t_build = time.perf_counter() - t1

        t2 = time.perf_counter()
        response = create_onebill_order(session, ONEBILL_ORDER_URL, payload)
        t_net = time.perf_counter() - t2

        order_id = response.get("orderId", "unknown")
        status = "success"
        logger.info(
            f"  [OK] {account_code} / {subscription_id} (shipAddId={ship_add_id}, "
            f"OneBill orderId={order_id}) — lookup={t_lookup*1000:.0f}ms "
            f"build={t_build*1000:.0f}ms net={t_net*1000:.0f}ms"
        )

    except Exception as e:
        error = str(e)
        logger.error(f"  [FAIL] {account_code} / {subscription_id} — {error}")

    return {
        "AccountCode":       account_code,
        "SubscriptionUSN":   subscription_id,
        "shipAddId":         ship_add_id,
        "status":            status,
        "onebill_order_id":  order_id,
        "error":             error,
        "elapsed_lookup_ms": round(t_lookup * 1000, 1),
        "elapsed_build_ms":  round(t_build * 1000, 1),
        "elapsed_net_ms":    round(t_net   * 1000, 1),
    }

## 7. Parallel Migration Loop

Fans the subscription list out across `MAX_WORKERS` threads. The HTTP session
is shared (with a pool sized to match worker count) so connections are
reused rather than re-established for every request.

Each subscription now does at least one GET (address lookup, skipped if that
account is already cached) plus one POST, so overall request volume is
somewhat higher than the account migration — the cache keeps this to one GET
per unique account rather than one per subscription. OneBill's sandbox has
been observed to throttle past ~20 concurrent writers — increase
`MAX_WORKERS` cautiously.

In [ ]:
def migrate(df: pd.DataFrame, max_workers: int = MAX_WORKERS) -> pd.DataFrame:
    """Migrate every subscription in df to OneBill in parallel."""
    session = requests.Session()
    adapter = requests.adapters.HTTPAdapter(
        pool_connections=max_workers,
        pool_maxsize=max_workers,
    )
    session.mount("https://", adapter)
    session.headers.update({
        "proxy_accountNumber": ONEBILL_PROXY_ACCT,
        "Content-Type":        "application/json",
    })

    rows  = [row for _, row in df.iterrows()]
    total = len(rows)
    results: list[dict] = []

    logger.info(f"Starting migration of {total:,} subscriptions with {max_workers} workers...")
    wall_start = time.perf_counter()

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(migrate_row, row, session): row["SubscriptionUSN"]
            for row in rows
        }

        for i, future in enumerate(as_completed(futures), start=1):
            results.append(future.result())

            if i % 50 == 0 or i == total:
                ok   = sum(1 for r in results if r["status"] == "success")
                fail = sum(1 for r in results if r["status"] == "failed")
                logger.info(f"Progress: {i}/{total} — {ok} ok, {fail} failed")

    wall_elapsed = time.perf_counter() - wall_start
    results_df = pd.DataFrame(results)
    success = (results_df["status"] == "success").sum()
    failed  = (results_df["status"] == "failed").sum()

    logger.info(
        f"Migration done in {wall_elapsed:.1f}s — "
        f"{success} succeeded, {failed} failed. (log: {log_filename})"
    )

    # --- Profiling summary
    print("\n=== Profiling Summary ===")
    print(f"Total wall time:          {wall_elapsed:.1f}s")
    if wall_elapsed > 0:
        print(f"Throughput:               {total / wall_elapsed:.1f} subscriptions/s")
    print(f"Succeeded / Failed:       {success} / {failed}")
    print(f"Unique accounts cached:   {len(_ship_address_cache):,}")

    return results_df

## 8. Run the Migration

In [ ]:
results_df = migrate(df_subscriptions)

failures = results_df[results_df["status"] == "failed"]
print(f"\nFailed rows ({len(failures):,}):")
failures.head(20)

### Export failures

In [ ]:
out_path = f'Failed_Subscription_Migrations_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
failures.to_csv(out_path, index=False)
print(f"Wrote {len(failures):,} failures to {out_path}")

### Error summary

In [ ]:
if not failures.empty:
    error_summary = (
        failures.groupby("error")
        .agg(count=("SubscriptionUSN", "size"),
             example_account=("AccountCode", "first"),
             example_subscription=("SubscriptionUSN", "first"))
        .sort_values("count", ascending=False)
        .reset_index()
    )
    print(f"Distinct error messages: {len(error_summary):,}")
else:
    print("No failures to summarise.")
    error_summary = pd.DataFrame()
error_summary